In [2]:
!pip install Sastrawi openpyxl scikit-learn pandas numpy nltk seaborn matplotlib gensim

import random
import os
import re
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from gensim.models import FastText
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import normalize
import seaborn as sns
import matplotlib.pyplot as plt

nltk.download('punkt')
nltk.download('punkt_tab')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 9.6 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# ==============================================================================
# SISTEM PREPARASI LINGKUNGAN KAGGLE (Wajib di Jalankan Paling Awal)
# ==============================================================================
# Menginstal library Sastrawi otomatis karena belum tersedia bawaan di Kaggle
!pip install Sastrawi -q

# ==============================================================================
# TAHAP 1: PREPROCESSING (Pembersihan Data Eksperimen)
# ==============================================================================
import random
import os
import re
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengunci Seed lingkungan demi menjamin konsistensi hasil (Reproducibility)
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)

# Mengunduh paket tokenisasi NLTK secara aman di backend Kaggle
nltk.download('punkt', quiet=True)

# 1. Memuat Dataset dari Direktori Input Kaggle
# Catatan: Kita arahkan ke folder input Kaggle sesuai struktur dataset Anda
kaggle_path = '/kaggle/input/datasets/fauzanrass/mybca'

# Cek ekstensi file di dalam folder Kaggle secara otomatis untuk mencegah error pembacaan
try:
    if os.path.isdir(kaggle_path):
        files = os.listdir(kaggle_path)
        target_file = os.path.join(kaggle_path, files[0])
    else:
        target_file = kaggle_path

    print(f"📂 Mendeteksi file data di: {target_file}")
    
    # Membaca berdasarkan format file yang ditemukan
    if target_file.endswith('.xlsx') or target_file.endswith('.xls'):
        df = pd.read_excel(target_file, sheet_name='Total')
    else:
        df = pd.read_csv(target_file)
        
    df = df[['Komentar', 'Sentimen']].dropna()
    print(f"📊 Dataset Berhasil Dimuat! Jumlah data awal: {len(df)} ulasan.")
except Exception as e:
    print(f"❌ Gagal memuat file. Silakan periksa kembali letak path data Anda. Pesan Error: {e}")

# 2. Konfigurasi Kamus Negasi & Stopwords Bahasa Indonesia
factory_stop = StopWordRemoverFactory()
stop_words = set(factory_stop.get_stop_words())

# Memisahkan kata negasi agar bobot sentimen tidak hilang/terbalik saat ekstraksi fitur
negation_words = {
    'tidak', 'bukan', 'belum', 'jangan', 'kurang', 'tanpa', 'gagal', 'batal',
    'ga', 'gak', 'ngga', 'nggak', 'engga', 'enggak', 'gx', 'g', 'tdk',
    'ndak', 'nda', 'kagak', 'kgk', 'kaga', 'tak', 'gk', 'nggk', 'ngak',
    'gabisa', 'gakbisa', 'gbisa', 'gkbisa', 'gakan', 'gaakan', 'nggakbisa',
    'blm', 'belom', 'blom', 'lom', 'lum', 'belon',
    'bkn', 'bukang', 'bkan', 'jgn', 'jan', 'krg', 'krng'
}
stop_words = stop_words - negation_words

# Menambahkan noise kata khas ulasan aplikasi perbankan ke dalam daftar stopwords
custom_stopwords = {'yg', 'nya', 'sih', 'deh', 'dong', 'kok', 'kan', 'min', 'bca', 'mybca', 
                    'aku', 'saya', 'dan', 'di', 'ke', 'dari', 'buat', 'untuk', 'ini', 'itu'}
stop_words = stop_words.union(custom_stopwords)

# 3. Kamus Normalisasi Slang & Penanganan Typo Bahasa Gaul
slang_dict = {
    "ga": "tidak", "gak": "tidak", "ngga": "tidak", "nggak": "tidak", "engga": "tidak", "enggak": "tidak", "gx": "tidak", "g": "tidak",
    "tdk": "tidak", "ndak": "tidak", "nda": "tidak", "kagak": "tidak", "kgk": "tidak", "kaga": "tidak", "tak": "tidak", "gk": "tidak",
    "nggk": "tidak", "ngak": "tidak", "gabisa": "tidak bisa", "gakbisa": "tidak bisa", "gbisa": "tidak bisa", "gkbisa": "tidak bisa",
    "gakan": "tidak akan", "gaakan": "tidak akan", "nggakbisa": "tidak bisa", "blm": "belum", "belom": "belum", "blom": "belum",
    "lom": "belum", "lum": "belum", "belon": "belum", "bkn": "bukan", "bukang": "bukan", "bkan": "bukan", "jgn": "jangan", "jan": "jangan",
    "krg": "kurang", "krng": "kurang", "yg": "yang", "tp": "tapi", "tf": "transfer", "kalo": "kalau", "bgs": "bagus", "apk": "aplikasi",
    "lemot": "lambat", "eror": "error", "ngelek": "lag", "bener": "benar", "mulu": "terus", "dongo": "bodoh", "hbs": "habis", "bs": "bisa",
    "rek": "rekening", "mbanking": "m-banking", "pake": "pakai", "udah": "sudah", "dpt": "dapat", "dr": "dari", "krn": "karena",
    "karna": "karena", "klo": "kalau", "bgt": "banget", "dgn": "dengan", "pdhl": "padahal", "gmn": "bagaimana", "gmana": "bagaimana"
}

# 4. Fungsi Utama Pembersihan Teks (Text Cleansing Pipeline)
def preprocess_text(text):
    text = str(text).lower() # Case Folding
    text = re.sub(r'http\S+|www\S+|@[^\s]+|#\S+', '', text) # Hapus URL, Mention, Hashtag
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1', text) # Normalisasi kata memanjang (misal: "bagaaaus" -> "bagus")
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Hapus angka, tanda baca, simbol emonji
    
    tokens = word_tokenize(text)
    cleaned_tokens = []
    
    for t in tokens:
        # Melakukan konversi kata baku lewat Kamus Slang
        t = slang_dict.get(t, t)
        mapped_words = t.split()
        
        for word in mapped_words:
            # Seleksi kata non-stopword dan menyingkirkan huruf tunggal sisa pembersihan
            if word not in stop_words and len(word) > 1:
                cleaned_tokens.append(word)
                
    return cleaned_tokens

# 5. Eksekusi Proses Transformasi Data
print("⏳ Menjalankan Pipeline Preprocessing pada Dataset...")
df['Clean_Tokens'] = df['Komentar'].apply(preprocess_text)
df['Clean_Text'] = df['Clean_Tokens'].apply(lambda x: ' '.join(x))

# Menyaring dan membuang ulasan yang berubah menjadi string kosong setelah dibersihkan
df = df[df['Clean_Text'].str.strip().astype(bool)]
print(f"✅ Tahap 1 Selesai! Sisa baris data valid untuk eksperimen: {len(df)} ulasan.")

📂 Mendeteksi file data di: /kaggle/input/datasets/unpredictt/mybcaaa/mybca.xlsx
📊 Dataset Berhasil Dimuat! Jumlah data awal: 9508 ulasan.
⏳ Menjalankan Pipeline Preprocessing pada Dataset...
✅ Tahap 1 Selesai! Sisa baris data valid untuk eksperimen: 9445 ulasan.


BASELINE

In [9]:
# ==============================================================================
# TAHAP 2: EVALUASI BASELINE (TF-IDF) - AKURASI & F1-SCORE
# ==============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate
import pandas as pd
import numpy as np

print("⏳ 1. Mengekstrak Fitur TF-IDF (Baseline)...")
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_tfidf = tfidf_vectorizer.fit_transform(df['Clean_Text'])
y = df['Sentimen'].values

# Daftar parameter
k_list = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
distance_metrics = ['euclidean', 'manhattan', 'cosine']
results_baseline = []

print("⏳ 2. Mencari Kombinasi K & Metrik Optimal (5-Fold CV)...")
for metric in distance_metrics:
    for k in k_list:
        knn_model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        
        # Menggunakan cross_validate untuk mengambil beberapa metrik sekaligus
        scoring = ['accuracy', 'f1_macro']
        scores = cross_validate(knn_model, X_tfidf, y, cv=5, scoring=scoring)
        
        results_baseline.append({
            'Metrik_Jarak': metric,
            'Nilai_K': k,
            'Akurasi': scores['test_accuracy'].mean(),
            'F1_Score': scores['test_f1_macro'].mean()
        })
    print(f"✅ Selesai mengevaluasi metrik: {metric}")

# 3. Merangkum Hasil
df_results = pd.DataFrame(results_baseline)

# Menampilkan tabel perbandingan
print("\n" + "="*80)
print("📊 TABEL PERBANDINGAN: AKURASI & F1-SCORE")
print("="*80)
print(df_results.sort_values(by='Akurasi', ascending=False).round(4).to_string(index=False))

# Identifikasi Konfigurasi Terbaik (Berdasarkan Akurasi)
best_config = df_results.loc[df_results['Akurasi'].idxmax()]

print("\n" + "="*80)
print("🎯 KESIMPULAN TAHAP 2 (BASELINE)")
print("="*80)
print(f"Konfigurasi Terbaik: Metrik '{best_config['Metrik_Jarak']}' dengan K = {int(best_config['Nilai_K'])}")
print(f"Akurasi Tertinggi : {best_config['Akurasi']*100:.2f}%")
print(f"F1-Score Tertinggi: {best_config['F1_Score']:.4f}")
print("="*80)

⏳ 1. Mengekstrak Fitur TF-IDF (Baseline)...
⏳ 2. Mencari Kombinasi K & Metrik Optimal (5-Fold CV)...
✅ Selesai mengevaluasi metrik: euclidean
✅ Selesai mengevaluasi metrik: manhattan
✅ Selesai mengevaluasi metrik: cosine

📊 TABEL PERBANDINGAN: AKURASI & F1-SCORE
Metrik_Jarak  Nilai_K  Akurasi  F1_Score
      cosine       21   0.9169    0.9169
      cosine       15   0.9159    0.9159
      cosine       19   0.9158    0.9158
      cosine       13   0.9157    0.9157
      cosine       17   0.9155    0.9155
      cosine       11   0.9142    0.9142
      cosine        9   0.9124    0.9124
      cosine        7   0.9101    0.9101
      cosine        5   0.9095    0.9094
      cosine        3   0.9020    0.9019
      cosine        1   0.8764    0.8764
   euclidean        1   0.6358    0.5983
   euclidean        3   0.5900    0.5227
   euclidean        5   0.5665    0.4807
   euclidean        7   0.5539    0.4550
   manhattan        1   0.5457    0.4292
   euclidean        9   0.5431    0.4319

OPTIMASI FAST TEXT

In [12]:
# ==============================================================================
# TAHAP 3: OPTIMASI PARAMETER FASTTEXT (Dimensi 100-500 & Window 1-7)
# ==============================================================================
from gensim.models import FastText
from sklearn.preprocessing import normalize
from sklearn.model_selection import cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import pandas as pd

# Fungsi pembobotan vektor
def get_weighted_ft(tokens, model, tf_dict, size):
    vec = np.zeros(size)
    w_sum = 0
    for w in tokens:
        if w in model.wv.key_to_index and w in tf_dict:
            w_val = tf_dict[w]
            vec += model.wv[w] * w_val
            w_sum += w_val
    return vec / w_sum if w_sum > 0 else vec

# Parameter
vector_sizes = [100, 200, 300, 400, 500]
window_sizes = [1, 2, 3, 4, 5, 6, 7]
k_tetap = int(best_config['Nilai_K']) 

results_ft = []

print(f"⏳ Memulai Optimasi FastText (K={k_tetap}, Cosine, Akurasi & F1-Score)...")

for v_size in vector_sizes:
    for w_size in window_sizes:
        # A. Latih Model
        model_ft = FastText(sentences=df['Clean_Tokens'].tolist(),
                            vector_size=v_size, window=w_size,
                            min_count=2, workers=1, sg=1, epochs=20, seed=42)
        
        # B. Ekstraksi Fitur
        tfidf_temp = TfidfVectorizer().fit(df['Clean_Text'])
        tf_dict = dict(zip(tfidf_temp.get_feature_names_out(), tfidf_temp.idf_))
        X_ft_temp = normalize(np.array([get_weighted_ft(t, model_ft, tf_dict, v_size) for t in df['Clean_Tokens']]))
        
        # C. Evaluasi (Menggunakan cross_validate untuk Akurasi & F1-Macro)
        knn_ft = KNeighborsClassifier(n_neighbors=k_tetap, weights='distance', metric='cosine')
        scoring = ['accuracy', 'f1_macro']
        scores = cross_validate(knn_ft, X_ft_temp, y, cv=5, scoring=scoring)
        
        results_ft.append({
            'Dimensi': v_size, 
            'Window': w_size, 
            'Akurasi': scores['test_accuracy'].mean(),
            'F1_Score': scores['test_f1_macro'].mean()
        })
        print(f"✅ Dimensi: {v_size:3} | Window: {w_size} -> Akurasi: {scores['test_accuracy'].mean()*100:.2f}% | F1: {scores['test_f1_macro'].mean():.4f}")

# 3. Rekapitulasi
df_ft_results = pd.DataFrame(results_ft)

print("\n" + "="*80)
print("📊 HASIL OPTIMASI FASTTEXT TERBAIK (SORTED BY AKURASI)")
print("="*80)
# Menampilkan 10 hasil teratas
print(df_ft_results.sort_values(by='Akurasi', ascending=False).head(10).round(4).to_string(index=False))

best_ft = df_ft_results.loc[df_ft_results['Akurasi'].idxmax()]

print("\n" + "="*80)
print(f"🎯 KESIMPULAN TAHAP 3:")
print(f"Konfigurasi Terbaik: Dimensi {int(best_ft['Dimensi'])}, Window {int(best_ft['Window'])}")
print(f"Akurasi Puncak  : {best_ft['Akurasi']*100:.2f}%")
print(f"F1-Score Terbaik: {best_ft['F1_Score']:.4f}")
print("="*80)

⏳ Memulai Optimasi FastText (K=21, Cosine, Akurasi & F1-Score)...
✅ Dimensi: 100 | Window: 1 -> Akurasi: 91.48% | F1: 0.9147
✅ Dimensi: 100 | Window: 2 -> Akurasi: 91.76% | F1: 0.9175
✅ Dimensi: 100 | Window: 3 -> Akurasi: 91.75% | F1: 0.9174


KeyboardInterrupt: 

FAST TEXT KNN

In [10]:
# ==============================================================================
# TAHAP 5: OPTIMASI FASTTEXT MURNI (Dimensi 400, Window 6) - AKURASI & F1-SCORE
# ==============================================================================
from sklearn.model_selection import cross_validate

print("⏳ 1. Melatih Model FastText Murni (Dimensi 400, Window 6)...")
model_ft_pure = FastText(sentences=df['Clean_Tokens'].tolist(),
                         vector_size=400, window=6,
                         min_count=2, workers=1, sg=1, epochs=20, seed=42)

# Ekstraksi Vektor (Tanpa pembobotan TF-IDF, langsung rata-rata vektor kata)
def get_simple_mean_ft(tokens, model, size):
    vec = np.zeros(size)
    count = 0
    for w in tokens:
        if w in model.wv.key_to_index:
            vec += model.wv[w]
            count += 1
    return vec / count if count > 0 else vec

X_ft_pure = normalize(np.array([get_simple_mean_ft(t, model_ft_pure, 400) for t in df['Clean_Tokens']]))

# Setup Eksperimen
k_ganjil = range(1, 22, 2) # [1, 3, 5, ..., 21]
metrics = ['euclidean', 'manhattan', 'cosine']
results_pure = []

print("⏳ 2. Menguji K (Ganjil) & Metrik Jarak...")
for m in metrics:
    for k in k_ganjil:
        knn = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=m)
        
        scoring = ['accuracy', 'f1_macro']
        scores = cross_validate(knn, X_ft_pure, y, cv=5, scoring=scoring)
        
        results_pure.append({
            'Metrik': m,
            'K': k,
            'Akurasi': scores['test_accuracy'].mean(),
            'F1_Score': scores['test_f1_macro'].mean()
        })
        print(f"Metrik: {m:10} | K={k:2} -> Akurasi: {scores['test_accuracy'].mean()*100:.2f}% | F1: {scores['test_f1_macro'].mean():.4f}")

# Rekapitulasi
df_pure = pd.DataFrame(results_pure)
print("\n" + "="*80)
print("📊 HASIL EVALUASI FASTTEXT MURNI (TOP 10)")
print("="*80)
print(df_pure.sort_values(by='Akurasi', ascending=False).head(10).round(4).to_string(index=False))

⏳ 1. Melatih Model FastText Murni (Dimensi 400, Window 6)...
⏳ 2. Menguji K (Ganjil) & Metrik Jarak...
Metrik: euclidean  | K= 1 -> Akurasi: 91.05% | F1: 0.9105
Metrik: euclidean  | K= 3 -> Akurasi: 92.45% | F1: 0.9245
Metrik: euclidean  | K= 5 -> Akurasi: 92.93% | F1: 0.9292
Metrik: euclidean  | K= 7 -> Akurasi: 92.97% | F1: 0.9297
Metrik: euclidean  | K= 9 -> Akurasi: 92.98% | F1: 0.9298
Metrik: euclidean  | K=11 -> Akurasi: 93.10% | F1: 0.9309
Metrik: euclidean  | K=13 -> Akurasi: 93.09% | F1: 0.9308
Metrik: euclidean  | K=15 -> Akurasi: 93.01% | F1: 0.9301
Metrik: euclidean  | K=17 -> Akurasi: 93.02% | F1: 0.9302
Metrik: euclidean  | K=19 -> Akurasi: 93.07% | F1: 0.9306
Metrik: euclidean  | K=21 -> Akurasi: 93.04% | F1: 0.9304
Metrik: manhattan  | K= 1 -> Akurasi: 91.17% | F1: 0.9117
Metrik: manhattan  | K= 3 -> Akurasi: 92.46% | F1: 0.9246
Metrik: manhattan  | K= 5 -> Akurasi: 92.95% | F1: 0.9295
Metrik: manhattan  | K= 7 -> Akurasi: 93.07% | F1: 0.9306
Metrik: manhattan  | K= 9 -

HYBRID KNN

In [11]:
# ==============================================================================
# TAHAP 4: PROPOSED METHOD (Hybrid FastText + TF-IDF Concatenation)
# ==============================================================================
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import cross_validate

print("⏳ 1. Menyiapkan Fitur FastText Terbaik (Dimensi 400, Window 6)...")
# Latih ulang dengan parameter juara (Dimensi 400, Window 6)
model_ft_final = FastText(sentences=df['Clean_Tokens'].tolist(),
                          vector_size=400, window=6,
                          min_count=2, workers=1, sg=1, epochs=20, seed=42)

# Ekstraksi Vektor FastText (400D)
tfidf_temp = TfidfVectorizer().fit(df['Clean_Text'])
tf_dict = dict(zip(tfidf_temp.get_feature_names_out(), tfidf_temp.idf_))
X_ft_final = normalize(np.array([get_weighted_ft(t, model_ft_final, tf_dict, 400) for t in df['Clean_Tokens']]))

print("⏳ 2. Menyiapkan Fitur TF-IDF Terseleksi (800 Fitur Terbaik)...")
tfidf_final = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_tfidf_raw = tfidf_final.fit_transform(df['Clean_Text'])
selector = SelectKBest(chi2, k=800)
X_tfidf_final = normalize(selector.fit_transform(X_tfidf_raw, y).toarray())

print("⏳ 3. Menggabungkan Fitur (Hybrid Concatenation: 1200 Dimensi)...")
# 400D FastText + 800D TF-IDF = 1200 Dimensi
X_hybrid = np.hstack((X_ft_final, X_tfidf_final))

# 4. Eksperimen Optimasi K (1-21) pada Ruang Fitur Hybrid
k_hybrid_test = range(1, 22) 
results_hybrid = []

print("⏳ 4. Menguji Performa Hybrid (Akurasi & F1-Score) dengan K=1-21...")
for k in k_hybrid_test:
    knn_hybrid = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='cosine')
    
    scoring = ['accuracy', 'f1_macro']
    scores = cross_validate(knn_hybrid, X_hybrid, y, cv=5, scoring=scoring)
    
    results_hybrid.append({
        'K': k, 
        'Akurasi': scores['test_accuracy'].mean(),
        'F1_Score': scores['test_f1_macro'].mean()
    })
    print(f"Hybrid KNN | K={k:2} -> Akurasi: {scores['test_accuracy'].mean()*100:.2f}% | F1: {scores['test_f1_macro'].mean():.4f}")

# 5. Rekapitulasi Hasil
df_hybrid = pd.DataFrame(results_hybrid)
best_h = df_hybrid.loc[df_hybrid['Akurasi'].idxmax()]

print("\n" + "="*80)
print("📊 RINGKASAN HASIL TAHAP 4 (METODE HIBRIDA - 1200 DIMENSI)")
print("="*80)
print(df_hybrid.round(4).to_string(index=False))

print(f"\n🎯 KESIMPULAN TAHAP 4:")
print(f"Kombinasi Hybrid Terbaik pada K = {int(best_h['K'])}")
print(f"Akurasi Puncak : {best_h['Akurasi']*100:.2f}%")
print(f"F1-Score Terbaik: {best_h['F1_Score']:.4f}")
print("="*80)

⏳ 1. Menyiapkan Fitur FastText Terbaik (Dimensi 400, Window 6)...
⏳ 2. Menyiapkan Fitur TF-IDF Terseleksi (800 Fitur Terbaik)...
⏳ 3. Menggabungkan Fitur (Hybrid Concatenation: 1200 Dimensi)...
⏳ 4. Menguji Performa Hybrid (Akurasi & F1-Score) dengan K=1-21...
Hybrid KNN | K= 1 -> Akurasi: 91.41% | F1: 0.9141
Hybrid KNN | K= 2 -> Akurasi: 91.39% | F1: 0.9139
Hybrid KNN | K= 3 -> Akurasi: 92.83% | F1: 0.9283
Hybrid KNN | K= 4 -> Akurasi: 92.81% | F1: 0.9281
Hybrid KNN | K= 5 -> Akurasi: 93.30% | F1: 0.9330
Hybrid KNN | K= 6 -> Akurasi: 93.34% | F1: 0.9334
Hybrid KNN | K= 7 -> Akurasi: 93.33% | F1: 0.9333
Hybrid KNN | K= 8 -> Akurasi: 93.31% | F1: 0.9331
Hybrid KNN | K= 9 -> Akurasi: 93.39% | F1: 0.9339
Hybrid KNN | K=10 -> Akurasi: 93.33% | F1: 0.9333
Hybrid KNN | K=11 -> Akurasi: 93.43% | F1: 0.9342
Hybrid KNN | K=12 -> Akurasi: 93.48% | F1: 0.9348
Hybrid KNN | K=13 -> Akurasi: 93.48% | F1: 0.9348
Hybrid KNN | K=14 -> Akurasi: 93.47% | F1: 0.9347
Hybrid KNN | K=15 -> Akurasi: 93.56% | 

In [4]:
!pip install scikit-fuzzy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 14.6 MB/s eta 0:00:00a 0:00:01


FUZZY FAST TEXT

In [5]:
from gensim.models import FastText
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, pairwise_distances
import numpy as np
import pandas as pd

# 1. Pastikan fitur X dan label y_encoded tersedia (Ekstraksi ulang agar aman)
print("⏳ Menyiapkan fitur FastText (400D)...")
model_ft = FastText(sentences=df['Clean_Tokens'].tolist(),
                    vector_size=400, window=6,
                    min_count=2, workers=1, sg=1, epochs=20, seed=42)

def get_simple_mean_ft(tokens, model, size):
    vec = np.zeros(size)
    count = 0
    for w in tokens:
        if w in model.wv.key_to_index:
            vec += model.wv[w]
            count += 1
    return vec / count if count > 0 else vec

# Membuat X dan y_encoded
X = normalize(np.array([get_simple_mean_ft(t, model_ft, 400) for t in df['Clean_Tokens']]))
le = LabelEncoder()
y_encoded = le.fit_transform(df['Sentimen'].values)

# 2. Fungsi Fuzzy k-NN (Algoritma Keller et al.)
def fuzzy_knn_predict(X_train, y_train, X_test, k, metric, m=2):
    n_classes = len(np.unique(y_train))
    y_pred = []
    
    # Menghitung jarak antar data
    dists_matrix = pairwise_distances(X_test, X_train, metric=metric)
    
    for i in range(len(X_test)):
        dists = dists_matrix[i]
        idx = np.argsort(dists)[:k]
        
        memberships = np.zeros(n_classes)
        for c in range(n_classes):
            numerator = 0
            denominator = 0
            for j in range(k):
                d_j = dists[idx[j]] if dists[idx[j]] > 1e-10 else 1e-10
                weight = 1 / (d_j ** (2 / (m - 1)))
                if y_train[idx[j]] == c:
                    numerator += weight
                denominator += weight
            memberships[c] = numerator / denominator if denominator > 0 else 0
        y_pred.append(np.argmax(memberships))
    return np.array(y_pred)

# 3. Eksekusi Eksperimen
k_values = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
metrics = ['euclidean', 'manhattan', 'cosine']
results_final = []

print("⏳ Menjalankan Fuzzy k-NN dengan 3 Metrik Jarak...")
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for metric in metrics:
    for k in k_values:
        accs, f1s = [], []
        for train_idx, test_idx in kf.split(X, y_encoded):
            y_p = fuzzy_knn_predict(X[train_idx], y_encoded[train_idx], X[test_idx], k, metric=metric)
            accs.append(accuracy_score(y_encoded[test_idx], y_p))
            f1s.append(f1_score(y_encoded[test_idx], y_p, average='macro'))
        
        results_final.append({'Metrik': metric, 'K': k, 'Akurasi': np.mean(accs), 'F1': np.mean(f1s)})
        print(f"Metrik: {metric:10} | K={k:2} -> Akurasi: {np.mean(accs)*100:.2f}%")

# 4. Hasil Akhir
print("\n" + "="*70)
print("📊 HASIL FINAL: FUZZY k-NN (ALGORITMA KELLER)")
print("="*70)
print(pd.DataFrame(results_final).sort_values(by=['Metrik', 'Akurasi'], ascending=[True, False]).round(4).to_string(index=False))

⏳ Menyiapkan fitur FastText (400D)...
⏳ Menjalankan Fuzzy k-NN dengan 3 Metrik Jarak...
Metrik: euclidean  | K= 3 -> Akurasi: 92.59%
Metrik: euclidean  | K= 5 -> Akurasi: 92.84%
Metrik: euclidean  | K= 7 -> Akurasi: 93.02%
Metrik: euclidean  | K= 9 -> Akurasi: 93.02%
Metrik: euclidean  | K=11 -> Akurasi: 93.01%
Metrik: euclidean  | K=13 -> Akurasi: 92.99%
Metrik: euclidean  | K=15 -> Akurasi: 93.01%
Metrik: euclidean  | K=17 -> Akurasi: 93.00%
Metrik: euclidean  | K=19 -> Akurasi: 93.00%
Metrik: euclidean  | K=21 -> Akurasi: 93.01%
Metrik: manhattan  | K= 3 -> Akurasi: 92.76%
Metrik: manhattan  | K= 5 -> Akurasi: 92.91%
Metrik: manhattan  | K= 7 -> Akurasi: 93.12%
Metrik: manhattan  | K= 9 -> Akurasi: 93.26%
Metrik: manhattan  | K=11 -> Akurasi: 93.18%
Metrik: manhattan  | K=13 -> Akurasi: 93.08%
Metrik: manhattan  | K=15 -> Akurasi: 93.20%
Metrik: manhattan  | K=17 -> Akurasi: 93.19%
Metrik: manhattan  | K=19 -> Akurasi: 93.14%
Metrik: manhattan  | K=21 -> Akurasi: 93.15%
Metrik: cosi

In [2]:
!pip install scikit-fuzzy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 4.3 MB/s eta 0:00:0000:0100:01


FUZZY HYBRID

In [12]:
# ==============================================================================
# TAHAP: HYBRID FUZZY k-NN (Implementasi Algoritma Keller pada X_hybrid)
# ==============================================================================
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd

# 1. Pastikan Label dalam bentuk numerik
le = LabelEncoder()
y_encoded = le.fit_transform(y) # y adalah array sentimen dari Tahap 4

# 2. Fungsi Fuzzy k-NN (Algoritma Keller)
def fuzzy_knn_keller(X_train, y_train, X_test, k, metric, m=2):
    n_classes = len(np.unique(y_train))
    y_pred = []
    
    # Menghitung jarak antar data menggunakan metrik yang dipilih
    dists_matrix = pairwise_distances(X_test, X_train, metric=metric)
    
    for i in range(len(X_test)):
        dists = dists_matrix[i]
        idx = np.argsort(dists)[:k]
        
        # Hitung derajat keanggotaan untuk setiap kelas
        memberships = np.zeros(n_classes)
        denominator = 0
        for j in range(k):
            d_j = dists[idx[j]] if dists[idx[j]] > 1e-10 else 1e-10
            denominator += (1 / d_j) ** (2 / (m - 1))
            
        for c in range(n_classes):
            numerator = 0
            for j in range(k):
                d_j = dists[idx[j]] if dists[idx[j]] > 1e-10 else 1e-10
                u_jc = 1 if y_train[idx[j]] == c else 0
                numerator += u_jc * ((1 / d_j) ** (2 / (m - 1)))
            
            memberships[c] = numerator / denominator if denominator > 0 else 0
            
        y_pred.append(np.argmax(memberships))
    return np.array(y_pred)

# 3. Eksperimen Hybrid Fuzzy k-NN dengan 3 Metrik Jarak
k_values = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
metrics = ['euclidean', 'manhattan', 'cosine']
results_hybrid_fuzzy = []

print("⏳ Menjalankan Hybrid Fuzzy k-NN (Algoritma Keller) pada 1200 Dimensi...")
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for metric in metrics:
    for k in k_values:
        accs, f1s = [], []
        for train_idx, test_idx in kf.split(X_hybrid, y_encoded):
            pred = fuzzy_knn_keller(X_hybrid[train_idx], y_encoded[train_idx], X_hybrid[test_idx], k, metric=metric)
            accs.append(accuracy_score(y_encoded[test_idx], pred))
            f1s.append(f1_score(y_encoded[test_idx], pred, average='macro'))
        
        results_hybrid_fuzzy.append({'Metrik': metric, 'K': k, 'Akurasi': np.mean(accs), 'F1': np.mean(f1s)})
        print(f"Hybrid Fuzzy | Metrik: {metric:10} | K={k:2} -> Akurasi: {np.mean(accs)*100:.2f}%")

# 4. Ringkasan Akhir
df_hf = pd.DataFrame(results_hybrid_fuzzy)
print("\n" + "="*70)
print("📊 HASIL FINAL: HYBRID FUZZY k-NN (ALGORITMA KELLER)")
print("="*70)
print(df_hf.sort_values(by=['Metrik', 'Akurasi'], ascending=[True, False]).round(4).to_string(index=False))

⏳ Menjalankan Hybrid Fuzzy k-NN (Algoritma Keller) pada 1200 Dimensi...
Hybrid Fuzzy | Metrik: euclidean  | K= 3 -> Akurasi: 90.30%
Hybrid Fuzzy | Metrik: euclidean  | K= 5 -> Akurasi: 89.91%
Hybrid Fuzzy | Metrik: euclidean  | K= 7 -> Akurasi: 89.51%
Hybrid Fuzzy | Metrik: euclidean  | K= 9 -> Akurasi: 89.00%
Hybrid Fuzzy | Metrik: euclidean  | K=11 -> Akurasi: 88.88%
Hybrid Fuzzy | Metrik: euclidean  | K=13 -> Akurasi: 88.72%
Hybrid Fuzzy | Metrik: euclidean  | K=15 -> Akurasi: 88.52%
Hybrid Fuzzy | Metrik: euclidean  | K=17 -> Akurasi: 88.30%
Hybrid Fuzzy | Metrik: euclidean  | K=19 -> Akurasi: 88.07%
Hybrid Fuzzy | Metrik: euclidean  | K=21 -> Akurasi: 87.82%
Hybrid Fuzzy | Metrik: manhattan  | K= 3 -> Akurasi: 92.24%
Hybrid Fuzzy | Metrik: manhattan  | K= 5 -> Akurasi: 92.73%
Hybrid Fuzzy | Metrik: manhattan  | K= 7 -> Akurasi: 93.17%
Hybrid Fuzzy | Metrik: manhattan  | K= 9 -> Akurasi: 93.35%
Hybrid Fuzzy | Metrik: manhattan  | K=11 -> Akurasi: 93.50%
Hybrid Fuzzy | Metrik: manha